# TraceletCodeAgent (direct_prompt strategy, n_samples=1) on GAIA -- Qwen3.5-9B

Same as `traceletReAct_directprompt_qwen359b.ipynb`, but with `n_samples=1` instead of 3. See `traceletReAct_directprompt_n1.ipynb` for the ablation rationale. Uses distinct `output_file`/`pickle_dir` names so it doesn't collide with the n_samples=3 run.

In [ ]:
import os
import sys
sys.path.insert(0, "../examples/open_deep_research")

from dotenv import load_dotenv
load_dotenv()

from smolagents import OpenAIModel

model_name = "Qwen/Qwen3.5-9B"
enable_thinking = False  # hardcoded (was interactive input()) so this notebook can run headlessly via nbconvert
# Together requires enable_thinking nested inside chat_template_kwargs for vLLM-served open-weight
# models (see naiveReAct.ipynb). client_kwargs timeout bounds a single API call so a stalled/hanging
# stream fails within 5 minutes instead of hanging indefinitely.
model = OpenAIModel(
    model_id=model_name,
    api_base="https://api.together.ai/v1/",
    api_key=os.environ["TOGETHER_API_KEY"],
    extra_body={"chat_template_kwargs": {"enable_thinking": enable_thinking}},
    client_kwargs={"timeout": 300.0},
)

In [ ]:
# Reuse the standard GAIA tool stack, same as naiveReAct.ipynb
from common_setup import build_tools

tools, ti_tool, visualizer = build_tools(model)

In [ ]:
from smolagents.monitoring import LogLevel
from smolagents.tracelet_agent import TraceletCodeAgent

agent = TraceletCodeAgent(
    tools=tools,
    model=model,
    max_steps=50,
    verbosity_level=LogLevel.INFO,
    additional_authorized_imports=["pandas", "numpy", "PIL", "json", "io", "zipfile", "csv", "openpyxl"],
    n_samples=1,
    skeleton_strategy="direct_prompt",  # model emits the sentinel-marked skeleton itself
    stream_outputs=True,  # Qwen models via TogetherAI reject non-streaming requests outright
)

In [ ]:
# Load GAIA validation set from HuggingFace
import pandas as pd
from common_setup import load_gaia_dataset

SET_TO_RUN = "validation"
eval_ds = load_gaia_dataset(set_to_run=SET_TO_RUN)

print(f"Loaded {len(eval_ds)} examples")

In [ ]:
from common_setup import evaluate_agent, question_scorer

results = evaluate_agent(
    agent,
    eval_ds,
    ti_tool,
    visualizer,
    n_samples=None,
    output_file=f"tracelet_direct_n1_react_{model_name}.jsonl",
    pickle_dir=f"tracelet_direct_n1_react_{model_name}",
)

In [ ]:
df = pd.DataFrame(results)
total = len(df)
correct = df["is_correct"].sum()

print("=== TraceletCodeAgent (direct_prompt, n_samples=1) GAIA Evaluation Results ===")
print(f"Overall accuracy:   {correct}/{total} = {correct/total:.1%}")
print(f"Avg time per question: {df['time_taken_seconds'].mean():.1f}s")
print(f"Avg steps per question: {df['num_steps'].mean():.1f}")

total_tokens = df["token_counts"].apply(lambda x: x.get("total_tokens", 0)).mean()
print(f"Avg total tokens per question: {total_tokens:,.0f}")

print(f"\nTool usage (total calls across all questions):")
tool_usage_df = pd.DataFrame(df["tool_usage"].tolist()).sum().sort_values(ascending=False)
for tool, count in tool_usage_df.items():
    if count > 0:
        print(f"  {tool}: {int(count)}")